# IRL Election Simulation

Runs voting-rule simulations on **every real election CSV** in `data/`.

**Candidate-count filter** (`CANDIDATE_FORKS`) restricts processing to files
whose number of candidates matches one of the listed values.  Set to `None`
to include all files regardless of candidate count.


## Parameters

In [ ]:
from __future__ import annotations

In [ ]:
# Folder containing election CSVs
DATA_FOLDER = "data"

# Candidate-count filter
# Keep only CSVs whose number of candidates is in this list.
# Set to None (or an empty list) to process every file in the folder.
CANDIDATE_FORKS: list[int] | None = [5, 6, 7, 8, 9]

# Voting rules
RULE_CODES: list[str] = [
    "AP_T05",
    "AP_T07",
    "AP_T0GE",
    "BALD",
    "BLAC",
    "BORD",
    "BUCK_I",
    "BUCK_R",
    "CAIR",
    "COOM",
    "COPE",
    "HARE",
    "KIMR",
    "L4DV",
    "L6DV",
    "MJ",
    "MMIN",
    "NANS",
    "PLU1",
    "PLU2",
    "RV",
    "SCHU",
    "STAR",
    "VETO",
    "WOOD",
    "YOUN",
    "AP_K2",
    "AP_K3",
    "AP_KRP",
    "DODG_C",
    "COND",
]

OUTPUT_BASE = "res"

COMPUTE_METRICS: bool = True

# Global reproducibility seed. Set to None to disable deterministic mode.
REPRO_SEED: int | None = 161

## 1 Scan the data folder

In [ ]:
import hashlib
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.notebook import tqdm

from vote_simulation.models.data_generation.data_instance import DataInstance
from vote_simulation.models.results.result_config import ResultConfig
from vote_simulation.models.results.series_result import SimulationSeriesResult
from vote_simulation.models.results.total_result import SimulationTotalResult
from vote_simulation.models.rules.registry import _ensure_profile
from vote_simulation.simulation.simulation import run_rules_on_instance


def clean_incomplete_data(profile: pd.DataFrame) -> pd.DataFrame:
    """Remove rows with incomplete data (ie incomplete row caracterized by empty cols)

    Args:
        profile (pd.DataFrame): matrix of preferences (n_voters, n_candidates)

    Returns:
        pd.DataFrame: matrix of preferences with incomplete rows removed
    """
    profile = profile.dropna(axis=0, how="any")
    return profile


def save_cleaned_data(profile: pd.DataFrame, folder_path: str = "output.csv") -> None:
    """Save cleaned data to a new CSV file"""
    profile.to_csv(folder_path, index=False)


def normalize_between_0_and_1(arr: np.ndarray) -> np.ndarray:
    """if value > 1 exist, divide by ten (one of two possible cases)"""
    if np.any(arr > 1):
        arr = arr / 10
    return arr


def deterministic_step_seed(
    base_seed: int,
    election_name: str,
    n_voters: int,
    n_candidates: int,
) -> int:
    """Build a stable per-election seed independent from run order."""
    payload = f"{base_seed}|{election_name}|{n_voters}|{n_candidates}".encode()
    digest = hashlib.blake2b(payload, digest_size=8).digest()
    return int.from_bytes(digest, "little") % (2**32 - 1)


def _load_election_csv(csv_path: str) -> DataInstance:
    """Load a voter candidate CSV into a DataInstance.

    Expected format: header row (first column = unnamed row index,
    remaining columns = candidate labels); one voter per subsequent row.
    """
    df = pd.read_csv(csv_path, index_col=0)
    profile = clean_incomplete_data(df)
    profile = _ensure_profile(profile.to_numpy(dtype=np.float64))
    return DataInstance.from_profile(profile, file_path=csv_path)


def _extract_data(
    data_folder: str,
    candidate_forks: list[int] | None = None,
) -> list[dict]:
    """Scan a folder for election CSVs, apply a candidate-count filter, and print a summary."""
    folder = Path(data_folder)
    all_csv_files = sorted(folder.glob("*.csv"))

    catalogue: list[dict] = []
    for csv_path in all_csv_files:
        df_head = pd.read_csv(csv_path, index_col=0, nrows=0)  # header only
        n_candidates = len(df_head.columns)
        n_voters = sum(1 for _ in open(csv_path)) - 1  # subtract header row
        catalogue.append(
            {
                "path": str(csv_path),
                "name": csv_path.stem,
                "n_voters": n_voters,
                "n_candidates": n_candidates,
            }
        )

    fork_set = set(candidate_forks) if candidate_forks else None
    selected = [e for e in catalogue if fork_set is None or e["n_candidates"] in fork_set]

    counts = Counter(e["n_candidates"] for e in selected)
    print(f"Total CSV files found : {len(catalogue)}")
    print(f"After filter          : {len(selected)} file(s)")
    print(f"By candidate count    : {dict(sorted(counts.items()))}")
    print()
    for e in selected:
        print(f"  {e['name']:40s}  voters={e['n_voters']:5d}  candidates={e['n_candidates']} ")

    return selected


selected = _extract_data(DATA_FOLDER, CANDIDATE_FORKS)

In [ ]:
def _clean_data(selected: list[dict]) -> pd.DataFrame:
    comparison_rows = []

    for entry in selected:
        csv_path = entry["path"]
        df_raw = pd.read_csv(csv_path, index_col=0)
        df_clean = clean_incomplete_data(df_raw)

        n_raw = len(df_raw)
        n_clean = len(df_clean)
        n_dropped = n_raw - n_clean
        pct_dropped = 100 * n_dropped / n_raw if n_raw > 0 else 0.0

        try:
            comparison_rows.append(
                {
                    "election": entry["name"],
                    "n_candidates": entry["n_candidates"],
                    "voters_raw": n_raw,
                    "voters_clean": n_clean,
                    "rows_dropped": n_dropped,
                    "pct_dropped (%)": round(pct_dropped, 2),
                    "min_rating_raw": df_raw.to_numpy(dtype=float).min(),
                    "max_rating_raw": df_raw.to_numpy(dtype=float).max(),
                    "min_rating_clean": df_clean.min(axis=None).min() if not df_clean.empty else float("nan"),
                    "max_rating_clean": df_clean.max(axis=None).max() if not df_clean.empty else float("nan"),
                    "has_missing_raw": bool(df_raw.isnull().values.any()),
                    "has_missing_clean": bool(df_clean.isnull().values.any()),
                }
            )
        except Exception as e:
            print(f"Error processing {entry['name']}: {e}")

    comparison_df = pd.DataFrame(comparison_rows).set_index("election")

    print("=== Raw vs Cleaned — overview ===")
    display(comparison_df)

    print("\n=== Numeric summary (raw voters / cleaned voters / rows dropped) ===")
    display(comparison_df[["voters_raw", "voters_clean", "rows_dropped", "pct_dropped (%)"]].describe())

    n_affected = (comparison_df["rows_dropped"] > 0).sum()
    total_dropped = comparison_df["rows_dropped"].sum()
    print(f"\n{n_affected}/{len(comparison_df)} elections had incomplete rows — {total_dropped} rows dropped in total.")

    return comparison_df


comparison_df = _clean_data(selected)

## 2 — Run simulations

Each selected CSV file → one `SimulationSeriesResult` (one step each), keyed by
its own `(gen_model, n_voters, n_candidates)`.  This lets `SimulationTotalResult`
filter and aggregate by either axis exactly like the synthetic workflow.


In [ ]:
def run_simulations(
    selected: list[dict],
    rule_codes: list[str],
    compute_metrics: bool = True,
    repro_seed: int | None = None,
) -> SimulationTotalResult:
    total_result = SimulationTotalResult()

    if repro_seed is not None:
        np.random.seed(repro_seed)

    for entry in tqdm(selected, desc="Elections", leave=True):
        n_c = entry["n_candidates"]
        n_v = entry["n_voters"]
        name = entry["name"]
        print(f"Name : {name}")
        if n_v == 0:
            print(f"  [SKIP] {entry['name']}: no voters found.")
            continue

        step_config = ResultConfig.single(
            gen_model=name,
            n_voters=n_v,
            n_candidates=n_c,
            rules_codes=rule_codes,
        )

        try:
            di = _load_election_csv(entry["path"])
        except Exception as exc:
            print(f"  [SKIP] {entry['name']}: {exc}")
            continue

        step_seed = None
        rng_state = None
        if repro_seed is not None:
            step_seed = deterministic_step_seed(repro_seed, name, n_v, n_c)
            rng_state = np.random.get_state()
            np.random.seed(step_seed)

        step = run_rules_on_instance(
            di,
            rule_codes,
            config=step_config,
            compute_metrics=compute_metrics,
        )

        if rng_state is not None:
            np.random.set_state(rng_state)

        # one series per (gen_model, n_voters, n_candidates) — merge if already exists
        try:
            existing = total_result.get_series("IRL", n_v, n_c)
            existing.add_step(step)
        except KeyError:
            series = SimulationSeriesResult()
            series.add_step(step)
            series.config = ResultConfig.single(
                gen_model=name,
                n_voters=n_v,
                n_candidates=n_c,
                n_iterations=1,
                rules_codes=rule_codes,
            )
            total_result.add_series(series)

    print(f"\nSimulation complete — {total_result.series_count} series")
    return total_result


total_result = run_simulations(selected, RULE_CODES, COMPUTE_METRICS, REPRO_SEED)

## 3 — Overview heatmap

Mean pairwise distance between rules across all candidate-count groups.

In [ ]:
# Rows = n_candidates groups, columns = n_voters (one per election)
total_result.plot_metric_heatmap(row_param="n_candidates", col_param="n_voters")

In [ ]:
total_result.plot_metrics_rules_matrix()

## 4 — Per-candidate-count analysis

For each group: distance matrix heatmap, 2-D & 3-D MDS projections, metrics matrix.

In [ ]:
base_path = Path(OUTPUT_BASE)

unique_n_c = sorted({e["n_candidates"] for e in selected})

for n_c in unique_n_c:
    sub = total_result.filter(n_candidates=n_c)
    if sub.series_count == 0:
        continue

    out_dir = base_path / f"c{n_c}"
    out_csv = out_dir / "csv"

    print(f"\n── {n_c} candidates ({sub.series_count} elections) ──────────────────")

    sub.plot_mean_distance_matrix(save_path=str(out_dir / f"c{n_c}_distance.png"))
    sub.export_mean_distance_matrix_csv(save_path=str(out_csv / f"c{n_c}_distance.csv"))
    sub.plot_rules_2d(save_path=str(out_dir / f"c{n_c}_rules_2d.png"))
    sub.plot_rules_3d(save_path=str(out_dir / f"c{n_c}_rules_3d.png"))
    if COMPUTE_METRICS:
        sub.plot_metrics_rules_matrix(save_path=str(out_dir / f"c{n_c}_metrics.png"))
        sub.export_metrics_rules_matrix_csv(save_path=str(out_csv / f"c{n_c}_metrics.csv"))

In [ ]:
total_result.plot_rules_2d(save_path=str(base_path / "rules_2d.png"))
total_result.plot_rules_3d(save_path=str(base_path / "rules_3d.png"))
total_result.export_mean_distance_matrix_csv(save_path=str(base_path / "mean_distance.csv"))
total_result.export_metrics_rules_matrix_csv(save_path=str(base_path / "metrics.csv"))
total_result.plot_mean_distance_matrix(save_path=str(base_path / "mean_distance.png"))
total_result.plot_metrics_rules_matrix(save_path=str(base_path / "metrics.png"))

In [ ]:
def analyze_results(
    total_result: SimulationTotalResult,
    rule_codes: list[str],
    base_path: Path,
    candidate_forks: list[int] | None = None,
) -> tuple[SimulationTotalResult, pd.DataFrame]:

    excluded = {"AP_T0GE", "VETO", "COND"}
    filtered_rules = [r for r in total_result.rules if r not in excluded]
    total_filtered = total_result.filter_rules(filtered_rules)

    total_filtered.plot_rules_2d(save_path=str(base_path / "rules_2d_no_APT0GE.png"))

    forks = candidate_forks or sorted({k.n_candidates for k in total_result.keys})
    to_output = {}
    for c in forks:
        filter_f = total_filtered.filter(n_candidates=c)
        for v in filter_f.voter_counts:
            fcv = filter_f.filter(n_voters=v)
            name = fcv.gen_models[0]
            m = fcv.mean_distance_matrix_frame().drop_duplicates()
            to_output[name] = len(m)

    print(to_output)

    df = pd.DataFrame(list(to_output.items()), columns=pd.Index(["name", "count"]))
    df.to_csv(base_path / "elected_candidates_count.csv", index=False)

    display(df.describe())
    display(df.value_counts(subset=["count"]))

    return total_filtered, df


total_filtered, df = analyze_results(total_result, RULE_CODES, base_path, CANDIDATE_FORKS)

In [ ]:
fin = total_filtered.filter(gen_model="election_AUT_2024")

fin.plot_mean_distance_matrix()

fin.plot_metrics_rules_matrix()
fin.plot_rules_2d()

CONDORCET_CONSISTENT_RULES = [
    "BALD",
    "BLAC",
    "COPE",
    "SCHU",
    "MMIN",
    "NANS",
    "YOUN",
    "DODG_C",
    "CAIR",
    "WOOD",
    "HARE",
]

# Keep only rules present in the current filtered set
cc_rules = [r for r in CONDORCET_CONSISTENT_RULES if r in CONDORCET_CONSISTENT_RULES]

print(f"Condorcet-consistent rules kept : {cc_rules}")

total_cc = fin.filter_rules(cc_rules)

total_cc.plot_mean_distance_matrix(save_path=str(base_path / "aout_cc_mean_distance.png"))
# total_cc.plot_rules_2d()
# total_cc.plot_metrics_rules_matrix()

In [ ]:
total_cc = total_filtered.filter_rules(cc_rules)

total_cc.plot_mean_distance_matrix(save_path=str(base_path / "cc_mean_distance.png"))
# total_cc.plot_rules_2d()
# total_cc.plot_metrics_rules_matrix()

In [ ]:
fin = total_filtered.filter(gen_model="BELW2019_cses")

fin.plot_mean_distance_matrix()

fin.plot_metrics_rules_matrix()
fin.plot_rules_2d()

# total_filtered.plot_metrics_rules_matrix(save_path=str(base_path / "metrics_no_APT0GE.png"))

In [ ]:
total_filtered.plot_mean_distance_matrix(show=False, save_path="mean_distance_no_APT0GE.png")
total_filtered.plot_rules_2d(show=False, save_path="rules_2d_no_APT0GE.png")

In [ ]:
total_result.filter(gen_model="election_AUT_2024").plot_mean_distance_matrix()

In [ ]:
# detail on 5 candidates

c5 = total_filtered.filter(n_candidates=5)

for v in c5.voter_counts:
    fcv = c5.filter(n_voters=v)
    name = fcv.gen_models[0]
    print(f"Name : {name}  n_candidates=5  n_voters={v}")

    m = fcv.mean_distance_matrix_frame()
    m = m.drop_duplicates()
    print(m)